In [1]:
import pandas as pd
import numpy as np
import json
import time
import ollama

In [2]:
landmarks_gdf = pd.read_csv("output/landmarks_with_neighborhoods.csv")

In [4]:
# # Define the model and batch size for classification
# MODEL = "qwen3.5:4b" 
# BATCH_SIZE = 25 


# def classify_batch(texts: list[str], categories: list[str]) -> list[str]:
#     """Classify a batch of texts into one of `categories`. Returns a list
#     of labels in the same order as `texts`."""

#     numbered = "\n".join(f"{i}: {t}" for i, t in enumerate(texts))
#     prompt = f"""Classify each numbered item into exactly one of these categories:
#                 {", ".join(categories)}

#                 If nothing fits well, use "Other".

#                 Items:
#                 {numbered}

#                 Respond with ONLY a JSON array of {len(texts)} strings (the category for
#                 each item, in order). No explanation, no markdown fences."""

#     resp = ollama.chat(
#         model=MODEL,
#         messages=[{"role": "user", "content": prompt}],
#         options={"temperature": 0},  # deterministic-ish output for classification
#     )

#     raw = resp["message"]["content"].strip()
#     print(raw)
#     raw = raw.removeprefix("```json").removeprefix("```").removesuffix("```").strip()

#     labels = json.loads(raw)
#     if len(labels) != len(texts):
#         raise ValueError(f"Expected {len(texts)} labels, got {len(labels)}")
#     return labels


# def classify_column(
#     df: pd.DataFrame,
#     text_col: str,
#     categories: list[str],
#     new_col: str = "category",
#     batch_size: int = BATCH_SIZE,
# ) -> pd.DataFrame:
#     """Adds a classification column to df by batching rows into fewer API calls."""

#     df = df.copy()
#     all_labels: list[str] = []
#     texts = df[text_col].astype(str).tolist()

#     for start in range(0, len(texts), batch_size):
#         batch = texts[start:start + batch_size]
#         for attempt in range(3):
#             try:
#                 labels = classify_batch(batch, categories)
#                 break
#             except Exception as e:
#                 print(f"Batch {start}-{start+len(batch)} failed ({e}), retrying...")
#                 time.sleep(2 ** attempt)
#         else:
#             labels = ["Error"] * len(batch)
#         all_labels.extend(labels)
#         print(f"Classified rows {start}-{start + len(batch)} / {len(texts)}")

#     df[new_col] = all_labels
#     return df


# if __name__ == "__main__":
#     df = pd.DataFrame({
#         "feedback": [
#             "The app crashes every time I open the camera.",
#             "Loved the new checkout flow, so much faster!",
#             "Can you add dark mode please?",
#             "Support never responded to my ticket.",
#         ]
#     })

#     # categories = ["Bug Report", "Positive Feedback", "Feature Request", "Support Complaint"]

#     # result = classify_column(df, text_col="feedback", categories=categories)
#     # print(result)
#     # result.to_csv("classified_output.csv", index=False)

In [5]:
# categories = ["Religious Site", "Library", "Museum", "Park", 
#             "Government Building", "Residential Building", 
#             "Commercial Building", "Fire Station", "Police Station", 
#             "School", "Hospital", "Civic Infrastructure", "Other"]

# result = classify_column(dflandmarks_gdf, text_col="LM_NAME", categories=categories)
# print(result)
# result.to_csv("classified_output.csv", index=False)

In [6]:
categories = ["Religious Site", "Library", "Museum", "Park", 
            "Government Building", "Residential Building", 
            "Commercial Building", "Fire Station", "Police Station", 
            "School", "Hospital", "Civic Infrastructure", "Other"]

In [7]:
"""
Classify landmark titles into categories using sentence embeddings.
No API, no internet needed after the first run (model downloads once).

Install: pip install sentence-transformers pandas
"""

import pandas as pd
from sentence_transformers import SentenceTransformer, util

# 1. Load the embedding model (downloads once, then cached locally, ~80MB)
model = SentenceTransformer("all-MiniLM-L6-v2")

# 2. Define your categories
categories = ["Religious Site", "Library", "Museum", "Park", 
            "Government Building", "Residential Building", 
            "Commercial Building", "Fire Station", "Police Station", 
            "School", "Hospital", "Civic Infrastructure", "Other"]


titles = landmarks_gdf["LM_NAME"].tolist()      # <-- change "title" to your column name

# 4. Turn categories and titles into embeddings (vectors that capture meaning)
category_embeddings = model.encode(categories)
title_embeddings = model.encode(titles, show_progress_bar=True)

# 5. For each title, find the category it's most similar to
similarity_scores = util.cos_sim(title_embeddings, category_embeddings)
best_matches = similarity_scores.argmax(axis=1)  # index of best category per title

# 6. Add the result as a new column
landmarks_gdf["category"] = [categories[i] for i in best_matches]

# 7. Save it
landmarks_gdf.to_csv("landmarks_classified.csv", index=False)
print(landmarks_gdf[["LM_NAME", "category"]])

C:\Users\seanc\AppData\Roaming\Python\Python39\site-packages\tqdm\auto.py:21: TqdmWarning: IProgress not found. Please update jupyter and ipywidgets. See https://ipywidgets.readthedocs.io/en/stable/user_install.html
  from .autonotebook import tqdm as notebook_tqdm
c:\Users\seanc\anaconda3\envs\llamaenv\lib\site-packages\huggingface_hub\file_download.py:143: UserWarning: `huggingface_hub` cache-system uses symlinks by default to efficiently store duplicated files but your machine does not support them in C:\Users\seanc\.cache\huggingface\hub\models--sentence-transformers--all-MiniLM-L6-v2. Caching files will still work but in a degraded version that might require more space on your disk. This warning can be disabled by setting the `HF_HUB_DISABLE_SYMLINKS_WARNING` environment variable. For more details, see https://huggingface.co/docs/huggingface_hub/how-to-cache#limitations.
To support symlinks on Windows, you either need to activate Developer Mode or to run Python as an administrator

                                           LM_NAME              category
0                             Church of Saint Mary        Religious Site
1                           Public School 15 Annex                School
2                     Lithuanian Alliance Building   Government Building
3                         Lefcourt Clothing Center   Commercial Building
4                                  Barbey Building   Commercial Building
...                                            ...                   ...
1527                    Reverend David Moore House  Residential Building
1528                     121 Heberton Avenue House  Residential Building
1529  Public School 15 (Daniel D. Tompkins School)                School
1530                              Decker Farmhouse                  Park
1531                     105 Franklin Avenue House  Residential Building

[1532 rows x 2 columns]
